In [ ]:
# 11 — Bronze: ingest the raw JSON feed as-is with Auto Loader
# Every field is read as a string (types are enforced in silver), unexpected fields land in _rescued_data,
# and each row keeps its source file and ingest time. The checkpoint makes the step incremental:
# re-running only picks up files that have not been ingested yet.

In [ ]:
import os, sys, time
sys.path.insert(0, os.getcwd())          # pipeline_lib.py lives next to this notebook
import pipeline_lib as pl
from pyspark.sql import functions as F

dbutils.widgets.text("run_id", "manual")
RUN_ID = dbutils.widgets.get("run_id")          # tables are created by 10_export_raw

RAW_SCHEMA = """
  id STRING, reported_by_user_id STRING, handyman_user_id STRING, description STRING, incident_type STRING,
  urgency STRING, recommended_handyman_ids ARRAY<BIGINT>, agent_reasoning STRING, distance_km STRING,
  travel_time_minutes STRING, status STRING, rating STRING, feedback STRING,
  created_at STRING, assigned_at STRING, completed_at STRING, updated_at STRING,
  hours_worked STRING, amount_paid_eur STRING"""

In [ ]:
started = time.time()
query = (spark.readStream.format("cloudFiles")
         .option("cloudFiles.format", "json")
         .option("rescuedDataColumn", "_rescued_data")
         .option("pathGlobFilter", "*.json")
         .schema(RAW_SCHEMA)
         .load(pl.RAW_DIR)
         .select("*", F.col("_metadata.file_path").alias("_source_file"),
                 F.current_timestamp().alias("_ingested_at"))
         .writeStream
         .option("checkpointLocation", f"{pl.CHECKPOINT_DIR}/bronze")
         .trigger(availableNow=True)
         .toTable(pl.T_BRONZE))
query.awaitTermination()

new_rows = pl.rows_processed(query)
total = spark.table(pl.T_BRONZE).count()
pl.log_step(spark, RUN_ID, "11_bronze", started, rows_in=new_rows, rows_out=total,
            checks={"bronze_has_1m_rows": total >= 1_000_000})